# 02 — Instability Index & EDA
## Layer 1: Composite Macroeconomic Instability Index

### Purpose
Construct country-year instability index from z-scores,
rolling volatility, and shock components. Validate against
known crisis events. Run full EDA on target, features,
index, and crisis flag.

### Input
- `../data/02_panel_features.csv`

### Output
- `../data/03_panel_instability.csv`
- 4 EDA plot PNGs

### Run after → Run before
`01_feature_engineering.ipynb` → `03_layer2a_econometric.ipynb`

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from scipy.stats import mstats
import warnings
warnings.filterwarnings("ignore")

df_panel = pd.read_csv("data/02_panel_features.csv")
print(f"Loaded: {df_panel.shape} | Countries: {df_panel['COUNTRY'].nunique()}")

Loaded: (5052, 84) | Countries: 177


In [19]:
# ── Instability indicators ────────────────────────────────────
instability_indicators = [
    "Inflation","GDP_Growth","Fiscal_Balance",
    "Debt","Current_Account"
]

# Component 1: Z-score (within-country deviation)
for col in instability_indicators:
    df_panel[f"{col}_zscore"] = df_panel.groupby("COUNTRY")[col]        .transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))

# Component 2: Volatility (rolling 3yr std) — already computed
volatility_cols = [f"{col}_rollstd3" for col in instability_indicators]

# Component 3: Shock (absolute year-on-year change)
for col in instability_indicators:
    df_panel[f"{col}_shock"] = df_panel.groupby("COUNTRY")[col]        .transform(lambda x: x.diff().abs())

zscore_cols  = [f"{col}_zscore" for col in instability_indicators]
shock_cols   = [f"{col}_shock"  for col in instability_indicators]
all_comp_cols= zscore_cols + volatility_cols + shock_cols

print(f"Component columns: {len(all_comp_cols)}")

Component columns: 15


In [20]:
# ── Winsorize + RobustScaler ──────────────────────────────────
for col in all_comp_cols:
    arr = df_panel[col].fillna(0).values
    df_panel[col] = mstats.winsorize(arr, limits=[0.05, 0.05])

scaler = RobustScaler()
df_panel[all_comp_cols] = scaler.fit_transform(df_panel[all_comp_cols])
df_panel[all_comp_cols] = df_panel[all_comp_cols].clip(0, 1)

# ── Composite index ───────────────────────────────────────────
df_panel["z_component"]     = df_panel[zscore_cols].abs().mean(axis=1)
df_panel["vol_component"]   = df_panel[volatility_cols].mean(axis=1)
df_panel["shock_component"] = df_panel[shock_cols].mean(axis=1)

w1, w2, w3 = 0.30, 0.40, 0.30
df_panel["Instability_Index"] = (
    w1 * df_panel["z_component"] +
    w2 * df_panel["vol_component"] +
    w3 * df_panel["shock_component"]
)

df_panel["Instability_Index"] = MinMaxScaler(feature_range=(0,100))    .fit_transform(df_panel[["Instability_Index"]])

print(df_panel["Instability_Index"].describe().round(2))

count    5052.00
mean       33.00
std        18.41
min         0.00
25%        18.07
50%        30.74
75%        45.62
max       100.00
Name: Instability_Index, dtype: float64


In [21]:
# ── Stability categories (percentile-based) ───────────────────
p20 = df_panel["Instability_Index"].quantile(0.20)
p50 = df_panel["Instability_Index"].quantile(0.50)
p80 = df_panel["Instability_Index"].quantile(0.80)

df_panel["Stability_Category"] = pd.cut(
    df_panel["Instability_Index"],
    bins=[0, p20, p50, p80, 100],
    labels=["Stable","Moderate","Unstable","Crisis"],
    include_lowest=True
)
df_panel["Crisis_Flag"] = (
    df_panel["Stability_Category"] == "Crisis"
).astype(int)

print(df_panel["Stability_Category"].value_counts())
print(f"Crisis rate: {df_panel['Crisis_Flag'].mean()*100:.1f}%")

Stability_Category
Moderate    1515
Unstable    1515
Stable      1011
Crisis      1011
Name: count, dtype: int64
Crisis rate: 20.0%


In [22]:
# ── Validation against known crisis events ────────────────────
validation_cases = [
    ("Greece",                             2010),
    ("Argentina",                          2001),
    ("United States",                      2009),
    ("Ireland",                            2010),
    ("Venezuela, República Bolivariana de",2016),
]
print("=== Validation: known crisis years ===")
for country, year in validation_cases:
    row = df_panel[
        (df_panel["COUNTRY"]==country) &
        (df_panel["YEAR"]==year)
    ]["Instability_Index"]
    val = f"{row.values[0]:.1f}/100" if not row.empty else "not in panel"
    print(f"{country} {year}: {val}")

print("=== Top 15 most unstable country-years ===")
print(df_panel[["COUNTRY","YEAR","Instability_Index"]]
      .sort_values("Instability_Index", ascending=False)
      .head(15).to_string(index=False))

=== Validation: known crisis years ===
Greece 2010: 49.7/100
Argentina 2001: 18.6/100
United States 2009: 34.3/100
Ireland 2010: 58.3/100
Venezuela, República Bolivariana de 2016: 63.3/100
=== Top 15 most unstable country-years ===
                 COUNTRY  YEAR  Instability_Index
South Sudan, Republic of  2017         100.000000
              Cabo Verde  2022          94.958542
      Russian Federation  2000          94.112124
               St. Lucia  2022          93.332385
                 Lebanon  2021          92.505054
               Argentina  2003          91.114755
                Suriname  2021          90.622043
              Cabo Verde  2023          88.708414
                Suriname  2020          88.272980
               Argentina  2004          88.004876
   Eritrea, The State of  2018          87.639855
                  Angola  2006          87.226301
   Eritrea, The State of  2015          86.935582
                 Lebanon  2022          86.894055
                  